# 🏴 Ozz — HALctf Agent (Resilient Multi-Device Engine)
## DEF CON 34 AI Village — Autonomous Agent Pipeline

Pipeline limpo e linear otimizado para **Kaggle GPU (Tesla P100 / T4)**:
1. Instalar dependências
2. Clonar/configurar repositório
3. Subir Sandbox CTF Node.js (porta 3000)
4. Subir Servidor FastAPI LLM Qwen (porta 8000)
5. Health Check & Execução do Agente Ozz MNHI 3.5
6. Exportar relatório final

## 1. Setup — Instalar Dependências

In [ ]:
# Instalar dependências Python
!pip install -q transformers accelerate fastapi uvicorn requests pwntools beautifulsoup4 lxml pyjwt flask

# Instalar ferramentas auxiliares do sistema
!apt-get update -qq && apt-get install -y -qq nmap nikto gobuster netcat-openbsd curl wget sqlmap hydra smbclient mysql-client -qq 2>/dev/null || echo 'Ferramentas instaladas'

## 2. Configurar Diretórios e Repositório

In [ ]:
import os
working_dir = '/kaggle/working'
cache_dir = '/tmp/hf_cache'
os.makedirs(cache_dir, exist_ok=True)

if not os.path.exists(f'{working_dir}/ozz-halctf'):
    !git clone https://github.com/UNIFEI-CDA/ozz-halctf.git {working_dir}/ozz-halctf || echo 'Usando pasta local'

if os.path.exists(f'{working_dir}/ozz-halctf'):
    %cd {working_dir}/ozz-halctf
else:
    %cd {working_dir}
!ls -la

# Relatório final e exibição de logs do servidor Qwen
import json

# Imprimir logs internos do servidor FastAPI (hf_server.log) para diagnóstico de erros 500
try:
    if 'log_file' in globals() and not log_file.closed:
        log_file.flush()
    with open("/kaggle/working/hf_server.log", "r", encoding="utf-8", errors="ignore") as f:
        logs = f.read()
        print("\n======== LOGS INTERNOS DO SERVIDOR QWEN (hf_server.log) ========")
        print(logs[-4000:] if len(logs) > 4000 else logs)
        print("==================================================================\n")
except Exception as e:
    print(f"⚠️ Não foi possível ler hf_server.log: {e}")

sandbox_running = 'ctf_proc' in globals() and ctf_proc.poll() is None
agent_exit_code = agent_result.returncode if 'agent_result' in globals() else None
report = {
    "status": "SUCCESS" if agent_exit_code == 0 else "FINISHED",
    "agent": "Ozz MNHI 3.5",
    "model": "Qwen2.5-Coder-3B-Instruct",
    "sandbox_running": sandbox_running,
    "agent_exit_code": agent_exit_code,
}
with open("/kaggle/working/report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("✅ Relatório salvo em /kaggle/working/report.json!")
if 'server_proc' in globals():
    server_proc.terminate()
if 'ctf_proc' in globals() and ctf_proc.poll() is None:
    ctf_proc.terminate()
if 'ctf_log' in globals():
    ctf_log.close()


In [ ]:
# Subir Sandbox CTF Node.js na porta 3000 em background com resiliência
!if [ ! -d /kaggle/working/ctf-sandbox ]; then git clone https://github.com/kimdane/ctf.git /kaggle/working/ctf-sandbox || true; fi
import os, subprocess
!cd /kaggle/working/ctf-sandbox && (npm install --production --silent || true)
ctf_log = open('/kaggle/working/ctf.log', 'w', encoding='utf-8')
ctf_proc = subprocess.Popen(
    ['npm', 'start'],
    cwd='/kaggle/working/ctf-sandbox',
    env={**os.environ, 'PORT': '3000'},
    stdout=ctf_log,
    stderr=ctf_log
)
print('🚀 Processo da Sandbox CTF disparado em background na porta 3000!')

## 4. Criar e Iniciar Servidor FastAPI LLM Qwen 2.5 (Porta 8000)

In [ ]:
# Criar script hf_server.py com proteção de max_tokens, pad_token_id e logs detalhados de diagnóstico
server_script = '''
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Union
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn
import traceback

app = FastAPI()
model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
cache_dir = "/tmp/hf_cache"
print("📥 Carregando modelo Qwen 2.5 3B...")

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

current_device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if current_device == "cuda" else torch.float32

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        torch_dtype=dtype,
        device_map=current_device,
        trust_remote_code=True
    )
    print(f"✅ Modelo carregado com sucesso no dispositivo: {current_device}!")
except Exception as e:
    print(f"⚠️ Fallback para CPU devido a: {e}")
    current_device = "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True
    )

class ChatRequest(BaseModel):
    model: str
    messages: List[Dict[str, str]]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.3
    stop: Optional[Union[str, List[str]]] = None
    top_p: Optional[float] = None

@app.get("/v1/models")
def get_models():
    return {"data": [{"id": model_id}, {"id": "qwen2.5-coder-3b"}, {"id": "Qwen/Qwen2.5-Coder-7B-Instruct"}]}

@app.post("/v1/chat/completions")
def chat_completion(req: ChatRequest):
    global model, tokenizer, current_device
    try:
        try:
            prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
        except Exception as t_err:
            print(f"⚠️ Fallback no chat template: {t_err}")
            prompt = "\n".join([f"{m.get('role', 'user')}: {m.get('content', '')}" for m in req.messages])
        
        max_tok = min(req.max_tokens or 512, 512)
        print(f"🔍 [LLM Req] prompt length={len(prompt)}, max_tokens={max_tok}, device={current_device}")
        
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(current_device)
        temp = req.temperature if req.temperature is not None else 0.3
        gen_kwargs = {
            "max_new_tokens": max_tok,
            "do_sample": True if temp > 0 else False,
            "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id
        }
        if temp > 0:
            gen_kwargs["temperature"] = temp
        
        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return {"choices": [{"message": {"role": "assistant", "content": response_text}}]}
    except Exception as err:
        print(f"❌ Erro interno ao gerar resposta da LLM: {err}")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Internal generation error: {str(err)}")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("/kaggle/working/hf_server.py", "w", encoding="utf-8") as f:
    f.write(server_script)

# Iniciar servidor FastAPI em background
log_file = open("/kaggle/working/hf_server.log", "w", encoding="utf-8")
server_proc = subprocess.Popen(["python3", "/kaggle/working/hf_server.py"], stdout=log_file, stderr=log_file)
print("⚡ Servidor LLM FastAPI disparado na porta 8000 com proteção contra OOM e logs de diagnóstico!")

## 5. Health Check & Execução do Agente Ozz MNHI 3.5

In [ ]:
import time, requests, subprocess

# Health check: Aguardar servidor Qwen (porta 8000)
print('⏳ Aguardando servidor Qwen (http://localhost:8000/v1/models)...')
for i in range(60):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.status_code == 200:
            print('✅ Servidor Qwen pronto!')
            break
    except Exception:
        pass
    time.sleep(5)

# Health check: Aguardar Sandbox CTF (porta 3000)
print('⏳ Aguardando Sandbox CTF (http://localhost:3000)...')
for i in range(30):
    try:
        r = requests.get('http://localhost:3000', timeout=2)
        if r.status_code in [200, 301, 302, 403, 404]:
            print('✅ Sandbox CTF pronta!')
            break
    except Exception:
        pass
    time.sleep(2)

# Executar Agente autônomo Ozz MNHI 3.5
print('🚀 Disparando o agente Ozz MNHI 3.5...')
agent_result = subprocess.run(['python3', '-m', 'agent', 'http://localhost:3000'], check=False)
print(f'🏁 Agente finalizou com código de saída: {agent_result.returncode}')

## 6. Exportar Relatório Final em /kaggle/working

In [ ]:
# Relatório final
import json
sandbox_running = 'ctf_proc' in globals() and ctf_proc.poll() is None
agent_exit_code = agent_result.returncode if 'agent_result' in globals() else None
report = {
    "status": "SUCCESS" if agent_exit_code == 0 else "FINISHED",
    "agent": "Ozz MNHI 3.5",
    "model": "Qwen2.5-Coder-3B-Instruct",
    "sandbox_running": sandbox_running,
    "agent_exit_code": agent_exit_code,
}
with open("/kaggle/working/report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("✅ Relatório salvo em /kaggle/working/report.json!")
if 'server_proc' in globals():
    server_proc.terminate()
if 'ctf_proc' in globals() and ctf_proc.poll() is None:
    ctf_proc.terminate()
if 'ctf_log' in globals():
    ctf_log.close()